### REALHW229 小红的操作轨迹压缩
链接：[REALHW229 小红的操作轨迹压缩](https://www.nowcoder.com/practice/859a2ba75b92424cad9801eb40ec631c?tpId=390&tqId=11745410&sourceUrl=%2Fexam%2Foj%2Fta%3FtpId%3D37)

![题目描述](./desc.png)

In [ ]:
import sys

INF = int(1e9+1)


def solve():
    data = sys.stdin.buffer.read().split()
    it = iter(data)

    n = int(next(it))
    m = int(next(it))
    K = int(next(it))

    g = [[0] * m for _ in range(n)] # n * m
    dirs = {
        "U": [-1, 0],
        "D": [1, 0],
        "L": [0, -1],
        "R": [0, 1]
    }
    cur: list = [n // 2, m // 2]
    end: tuple | None = None
    while K:
        K -= 1
        move = str(next(it).decode())
        op = str(next(it).decode())
        param = int(next(it))

        ne = [cur[0] + dirs[move][0], cur[1] + dirs[move][1]]
        if not(ne[0] < 0 or ne[0] >= n or ne[1] < 0 or ne[1] >= m):
            cur = ne
        if op == "I":
            g[int(cur[0])][int(cur[1])] += param
        elif op == "D":
            g[int(cur[0])][int(cur[1])] -= param
        else:
            g[int(cur[0])][int(cur[1])] += param

    # 起始点、终点
    end = (cur[0], cur[1])
    start = (n // 2, m // 2)

    # 目标点收集
    targes = []
    for i in range(n):
        for j in range(m):
            if g[i][j]:
                targes.append((i, j))

    T = len(targes)
    allmask = (1 << T) - 1
    idx = {pos:k for k, pos in enumerate(targes)}


    # 状态表示：(x,y,mask)
    # x, y -> 最后到达点的坐标
    # mask 已经访问过的targe的2进制状态压缩表示
    # 距离原点的距离，可以用层序遍历，每一层距离+1
    init_state = (start[0], start[1], 0)

    from collections import deque
    dq = deque()
    dq.append(init_state)

    # 为了避免重复，需要将已经遍历过（已知最短路）的状态记录
    # 为什么是状态：因为如果是点的话，对于“T”型的路径，在分叉点，往右走后，要去左侧，就是直接左走，而非绕路
    # 这也同样解释了，为什么要使用状态
    # 两种方式路过某个点，但是方式1可能路过之前，已经路过了好几个targe点了
    visited = set()
    visited.add(init_state)

    # 记录当前层中的状态到初始状态的距离
    dist = 0

    while dq:
        # 遍历当前层中的状态
        for _ in range(len(dq)):
            x, y, mask = dq.popleft()

            # 终止的标准，当前点是终点，且经历过了所有targe
            if (x, y) == end and mask == allmask:
                print(dist)
                return

            # 遍历四个方向
            for dx, dy in dirs.values():
                nx, ny = x + dx, y + dy
                if nx < 0 or nx >= n or ny < 0 or ny >= m:
                    # 为什么不能直接跳过越界的操作，
                    # 题目中说了如果越界就在原地
                    # continue 
                    nx, ny = x, y
                
                # 更新 mask
                new_mask = mask
                if (nx, ny) in targes:
                    # 位运算，改变（xy,ny）对应的状态
                    # 使用 | ，因为之前可能已经遍历过了这个点
                    new_mask |= 1 << idx[(nx, ny)]
                
                # 新的 state
                new_state = (nx, ny, new_mask)

                # state 是否已经遍历过
                if new_state not in visited:
                    visited.add(new_state)
                    dq.append(new_state)
        # 别忘了，遍历玩一层，更新 dist
        dist += 1
        

solve()

"""
通过全部用例 运行时间 589ms 占用内存 28644KB
"""

存在的问题：

1. 思路半对，初始思路是使用 DFS 但是忽略了状态的表示，只考虑了点
    > 这也能提示我，能够搜索的不只有“实际上的点”，也可以是“不同的状态”
2. 同时，应该是 BFS 而非 DFS
    > BFS 能够让我的状态中少一个维度：距离，减轻思考
3. 2进制数进行状态压缩，我也没考虑到，一开始就直接用的set表示当前经历的路径
    > 一开始想的路径，有一点状态的意思，但是太过具体，也导致了无法压缩（经历同样的点，但是中间有重复进过的点，此时路径无法合并）